# 🏠 California Housing — EDA

**Objective:** Explore the California housing dataset to understand the factors driving median house prices and prepare for a regression model.

**Dataset:** 500 synthetic housing records with features: `med_income`, `house_age`, `avg_rooms`, `avg_bedrooms`, `population`, `avg_occupancy`, `latitude`, `longitude`, and target `price`.

**Notebook structure:**
1. Setup & Imports
2. Data Generation
3. Data Overview
4. Missing Values
5. Target (Price) Distribution
6. Correlation with Price
7. Feature Relationships
8. Insights & Conclusions

In [ ]:
# 1. Setup & Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Plot styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100

print("Setup complete.")

In [ ]:
# 2. Data Generation (synthetic, reproducible)
def generate_housing(n: int = 500, seed: int = 42) -> pd.DataFrame:
    """Generate a synthetic California housing-like dataset."""
    rng = np.random.default_rng(seed)
    med_income = rng.lognormal(mean=np.log(5), sigma=0.5, size=n)
    house_age = rng.uniform(0, 50, size=n)
    avg_rooms = rng.normal(6, 2, size=n).clip(2, 15)
    avg_bedrooms = avg_rooms * rng.uniform(0.15, 0.25, size=n)
    population = rng.poisson(500, size=n)
    avg_occupancy = rng.uniform(2, 6, size=n)
    latitude = rng.uniform(32.5, 42.0, size=n)
    longitude = rng.uniform(-124, -114, size=n)

    price = (
        -2.5 * med_income
        + 0.3 * house_age
        + 3.0 * avg_rooms
        - 2.0 * avg_bedrooms
        + 0.001 * population
        + 0.5 * avg_occupancy
        + 50 * (latitude - 37)
        - 30 * (longitude + 119)
        + rng.normal(0, 15, size=n)
        + 200
    ).clip(50, 500)

    df = pd.DataFrame({
        "med_income": med_income,
        "house_age": house_age,
        "avg_rooms": avg_rooms,
        "avg_bedrooms": avg_bedrooms,
        "population": population,
        "avg_occupancy": avg_occupancy,
        "latitude": latitude,
        "longitude": longitude,
        "price": price,
    })
    return df.round(2)


df = generate_housing(500)
print(f"Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns")
df.head(10)

In [ ]:
# 3. Data Overview
print("=== Data Types ===")
print(df.dtypes)
print("\n=== Statistical Summary ===")
df.describe().T

In [ ]:
# 4. Missing Values Check
missing = df.isnull().sum()
print("=== Missing Values ===")
print(missing[missing > 0] if missing.any() else "No missing values found. ✅")

# Duplicates
duplicates = df.duplicated().sum()
print(f"\nDuplicate rows: {duplicates}")

In [ ]:
# 5. Target (Price) Distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.histplot(df["price"], kde=True, ax=axes[0], color="steelblue", bins=30)
axes[0].set_title("Price Distribution", fontweight="bold")
axes[0].set_xlabel("Price ($100k)")

sns.boxplot(x=df["price"], ax=axes[1], color="steelblue")
axes[1].set_title("Price Boxplot", fontweight="bold")
axes[1].set_xlabel("Price ($100k)")

plt.tight_layout()
plt.show()

print(f"Price stats:\n{df['price'].describe().round(2)}")

In [ ]:
# 6. Correlation with Price
feature_cols = [c for c in df.columns if c != "price"]
corr_with_price = df[feature_cols].corrwith(df["price"]).sort_values(ascending=False)

print("=== Correlation with Price ===")
print(corr_with_price.round(3))

# Bar plot
plt.figure(figsize=(10, 5))
corr_with_price.plot(kind="bar", color=["#2ecc71" if v > 0 else "#e74c3c" for v in corr_with_price])
plt.title("Feature Correlation with Price", fontweight="bold")
plt.ylabel("Correlation")
plt.xticks(rotation=45, ha="right")
plt.axhline(0, color="black", linewidth=0.8, linestyle="--")
plt.tight_layout()
plt.show()

In [ ]:
# 7a. Scatter Plots — Key Features vs Price
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
scatter_pairs = [
    ("med_income", "Median Income vs Price"),
    ("avg_rooms", "Avg Rooms vs Price"),
    ("latitude", "Latitude vs Price"),
    ("longitude", "Longitude vs Price"),
]

for ax, (col, title) in zip(axes.flatten(), scatter_pairs):
    sns.scatterplot(data=df, x=col, y="price", ax=ax, alpha=0.6, color="steelblue")
    ax.set_title(title, fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
# 7b. Correlation Heatmap
plt.figure(figsize=(11, 9))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", mask=mask,
            linewidths=0.5, square=True, cbar_kws={"shrink": 0.8})
plt.title("Correlation Matrix (California Housing)", fontsize=14, fontweight="bold")
plt.show()

## 📌 Insights & Conclusions

1. **No missing values** — the dataset is clean and ready for modeling.
2. **Median income is the strongest driver** of house price (strong positive correlation).
3. **Geographic features matter** — latitude and longitude show meaningful correlation with price, reflecting regional price differences.
4. **Rooms & occupancy** have moderate positive relationships with price.
5. **Modeling implication:**
   - Linear models (Linear/Ridge Regression) are appropriate given the mostly linear relationships.
   - Consider feature scaling for `med_income`, `population`, and `avg_rooms`.
   - `avg_bedrooms` is highly correlated with `avg_rooms` — watch for multicollinearity.